# CDCR Facility Heat Risk Index

Computes a facility-level heat risk index for 31 CDCR state prisons following the Ovienmhada (2024) / VCP environmental risk framework.

**Risk = Hazard × Exposure × Vulnerability**

All sub-components are min-max normalized 0–1 before averaging within each component. Components are multiplied to produce a raw risk score, then normalized 0–100 cross-period (current and mid-century share the same normalization denominator).

## Components

| Component | Sub-components | Source |
|---|---|---|
| **Hazard** | `heat_hazard_idx_norm` / `heat_hazard_fut_idx_norm` (equal-weight mean of days_over_90, hotnights, AQI) | `data/heat_air_hazard.csv` via `tract_geoid` |
| **Exposure** | `days_indoor_above_78f_2025`, `ratio_indoor_to_outdoor`, `uhi_normalized`, `1 - pct_units_refrigeration` | `data/indoor_outdoor_heat_2025.csv` + `data/cdcr_facilities.csv` |
| **Vulnerability** | Medical acuity (P1+P2+medium), age >50, mental health (EOP), disability (DPP), race/POC | `data/cdcr_facilities.csv` |

**Note on facility coverage:** 31 of 34 state prisons have indoor exposure data. CAC, CVSP, and FWF are excluded (no indoor/outdoor heat model data available).

**Note on UHI nulls:** CCI and PVSP have no Benz & Burney (2021) UHI data — their tracts were classified as undeveloped. Imputed with system mean across 31 facilities.

**Note on PBSP ratio outlier:** PBSP (Pelican Bay, Crescent City coast) has `ratio_indoor_to_outdoor` = 15.75, driven by very few outdoor 78°F days (~4) in that coastal climate. This is physically plausible but will score PBSP at 1.0 on this sub-component. Flagged in output.

In [ ]:
import pandas as pd
import numpy as np

# Load data
cdcr = pd.read_csv('data/cdcr_facilities.csv')
hazard = pd.read_csv('data/heat_air_hazard.csv')
indoor = pd.read_csv('data/indoor_outdoor_heat_2025.csv')

print(f'cdcr_facilities rows: {len(cdcr)}')
print(f'heat_air_hazard rows: {len(hazard)}')
print(f'indoor_outdoor_heat rows: {len(indoor)}')

## 1. Build working dataset — 31 CDCR state prisons

In [ ]:
# Filter to CDCR state prisons (has cdcr_code, not fire camp)
state_prisons = cdcr[
    cdcr['cdcr_code'].notna() &
    (cdcr['cdcr_firecamp'].fillna(False) != True)
].copy()
print(f'State prisons in cdcr_facilities: {len(state_prisons)}')

# Inner join with indoor_outdoor — this restricts to the 31 with exposure data
# (excludes CAC, CVSP, FWF which have no indoor heat model data)
df = state_prisons.merge(indoor, on='cdcr_code', how='inner', suffixes=('', '_indoor'))
print(f'After join with indoor_outdoor: {len(df)} facilities')
print(f'Facilities: {sorted(df["cdcr_code"].tolist())}')

In [ ]:
# Join hazard via tract_geoid
# tract_geoid may be stored as float (e.g. 6077003406.0) — normalize to string
df['tract_geoid_str'] = df['tract_geoid'].astype(str).str.split('.').str[0]
hazard['GEOID_str'] = hazard['GEOID'].astype(str)

df = df.merge(
    hazard[['GEOID_str', 'heat_hazard_idx_norm', 'heat_hazard_fut_idx_norm', 'AQI_norm']],
    left_on='tract_geoid_str', right_on='GEOID_str', how='left'
)

print(f'Hazard join nulls: {df["heat_hazard_idx_norm"].isnull().sum()}')
print(f'heat_hazard_idx_norm range: {df["heat_hazard_idx_norm"].min():.1f} – {df["heat_hazard_idx_norm"].max():.1f}')
print(f'heat_hazard_fut_idx_norm range: {df["heat_hazard_fut_idx_norm"].min():.1f} – {df["heat_hazard_fut_idx_norm"].max():.1f}')

## 2. Normalization helper

In [ ]:
def minmax_norm(series):
    """Min-max normalize a series to 0–1 across the 31 facilities."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

## 3. Hazard component

Pre-computed in `data_sources/hazards/heat_hazard.ipynb` as equal-weight mean of:
- days_over_90_norm (Cal-Adapt, cross-period normalized)
- hotnights_norm (VCP 98th percentile, cross-period normalized)
- AQI_norm (mean of ozone, PM2.5, diesel percentiles from CalEnviroScreen)

Stored as 0–100; divide by 100 for 0–1 multiplication.

In [ ]:
df['hazard_current'] = df['heat_hazard_idx_norm'] / 100
df['hazard_midcentury'] = df['heat_hazard_fut_idx_norm'] / 100

print('Hazard component (0–1):')
print(df[['cdcr_code', 'hazard_current', 'hazard_midcentury']]
      .sort_values('hazard_midcentury', ascending=False).to_string(index=False))

## 4. Exposure component

4 equal-weight sub-components, each min-max normalized 0–1:
1. `days_indoor_above_78f_2025` — direct indoor heat burden
2. `ratio_indoor_to_outdoor` — building thermal amplification
3. `uhi_normalized` — geographic urban heat island (Benz & Burney 2021)
4. `1 - pct_units_refrigeration` — inverted AC coverage (high AC = low exposure)

In [ ]:
# Sub-component 1: indoor 78°F days
df['exp_indoor78'] = minmax_norm(df['days_indoor_above_78f_2025'])

# Sub-component 2: ratio indoor/outdoor
# PBSP outlier: ratio = 15.75 vs system max ~2.0 for all others
print('ratio_indoor_to_outdoor — top 5:')
print(df[['cdcr_code', 'ratio_indoor_to_outdoor']]
      .sort_values('ratio_indoor_to_outdoor', ascending=False).head(5).to_string(index=False))
df['exp_ratio'] = minmax_norm(df['ratio_indoor_to_outdoor'])

# Sub-component 3: UHI (already 0–1; impute 2 nulls with system mean)
uhi_nulls = df.loc[df['uhi_normalized'].isnull(), 'cdcr_code'].tolist()
print(f'\nuhi_normalized nulls: {uhi_nulls} — imputed with system mean')
uhi_mean = df['uhi_normalized'].mean()
df['uhi_filled'] = df['uhi_normalized'].fillna(uhi_mean)
df['exp_uhi'] = minmax_norm(df['uhi_filled'])

# Sub-component 4: inverted AC fraction
# pct_units_refrigeration is 0–1; invert so high AC = low exposure
df['ac_inverted'] = 1 - df['pct_units_refrigeration']
df['exp_noac'] = minmax_norm(df['ac_inverted'])

# Exposure score = equal-weight mean of 4 sub-components
exp_cols = ['exp_indoor78', 'exp_ratio', 'exp_uhi', 'exp_noac']
df['exposure_score'] = df[exp_cols].mean(axis=1)

print('\nExposure sub-components and score:')
print(df[['cdcr_code'] + exp_cols + ['exposure_score']]
      .sort_values('exposure_score', ascending=False).to_string(index=False))

## 5. Vulnerability component

5 equal-weight sub-components, each min-max normalized 0–1:
1. **Medical acuity** — sum of P1 + P2 + medium CCHCS risk tiers (% of facility population)
2. **Age** — % over 50
3. **Mental health** — % EOP designation
4. **Disability** — % DPP placement
5. **Race/POC** — % people of color (heat inequity + structural vulnerability)

In [ ]:
# Medical acuity = P1 + P2 + medium risk
df['medical_acuity'] = (
    df['cchcs_high_risk_p1_pct_2025'] +
    df['cchcs_high_risk_p2_pct_2025'] +
    df['cchcs_medium_risk_pct_2025']
)

vuln_inputs = {
    'medical_acuity': 'medical_acuity',
    'age_over_50':    'cchcs_age_over_50_pct_2025',
    'mental_health':  'cchcs_mental_health_eop_pct_2025',
    'disability':     'cchcs_dpp_pct_2025',
    'race_poc':       'race_peopleofcolor_pct',
}

# Check for nulls
print('Vulnerability input nulls:')
for label, col in vuln_inputs.items():
    n = df[col].isnull().sum()
    print(f'  {label} ({col}): {n} nulls')

# Normalize each sub-component
vuln_norm_cols = []
for label, col in vuln_inputs.items():
    norm_col = f'vuln_{label}'
    df[norm_col] = minmax_norm(df[col])
    vuln_norm_cols.append(norm_col)

# Vulnerability score = equal-weight mean
df['vulnerability_score'] = df[vuln_norm_cols].mean(axis=1)

print('\nVulnerability sub-components and score:')
print(df[['cdcr_code'] + vuln_norm_cols + ['vulnerability_score']]
      .sort_values('vulnerability_score', ascending=False).to_string(index=False))

## 6. Risk score

Raw Risk = Hazard × Exposure × Vulnerability

Normalized 0–100 **cross-period**: current and mid-century scores share the same min/max denominator, so they are directly comparable.

In [ ]:
df['raw_risk_current']    = df['hazard_current']    * df['exposure_score'] * df['vulnerability_score']
df['raw_risk_midcentury'] = df['hazard_midcentury'] * df['exposure_score'] * df['vulnerability_score']

# Cross-period normalization: min/max taken across both periods together
all_raw = pd.concat([df['raw_risk_current'], df['raw_risk_midcentury']])
raw_min, raw_max = all_raw.min(), all_raw.max()
print(f'Raw risk range (both periods): {raw_min:.4f} – {raw_max:.4f}')

df['risk_score_current']    = (df['raw_risk_current']    - raw_min) / (raw_max - raw_min) * 100
df['risk_score_midcentury'] = (df['raw_risk_midcentury'] - raw_min) / (raw_max - raw_min) * 100

print('\nRisk scores — mid-century ranked:')
print(df[['cdcr_code', 'hazard_midcentury', 'exposure_score', 'vulnerability_score',
          'risk_score_current', 'risk_score_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False)
      .round(2).to_string(index=False))

## 7. Output

Long format: two rows per facility (current + mid-century).
Saved to `data/CDCR_heat_risk_index.csv`.

In [ ]:
shared_cols = [
    'cdcr_code', 'name', 'average_2025_population',
    'exposure_score', 'vulnerability_score',
    'AQI_norm', 'ratio_indoor_to_outdoor', 'days_indoor_above_78f_2025',
    'uhi_normalized',
    # vulnerability raw inputs for interpretability
    'medical_acuity', 'cchcs_age_over_50_pct_2025',
    'cchcs_mental_health_eop_pct_2025', 'cchcs_dpp_pct_2025', 'race_peopleofcolor_pct',
]

current = df[shared_cols + ['hazard_current', 'risk_score_current']].copy()
current = current.rename(columns={'hazard_current': 'hazard_score', 'risk_score_current': 'risk_score'})
current['time_period'] = 'current'

midcentury = df[shared_cols + ['hazard_midcentury', 'risk_score_midcentury']].copy()
midcentury = midcentury.rename(columns={'hazard_midcentury': 'hazard_score', 'risk_score_midcentury': 'risk_score'})
midcentury['time_period'] = 'midcentury'

output = pd.concat([current, midcentury], ignore_index=True)
output = output.sort_values(['cdcr_code', 'time_period']).reset_index(drop=True)

# Round scores to 2 decimal places
score_cols = ['hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']
output[score_cols] = output[score_cols].round(2)

output.to_csv('data/CDCR_heat_risk_index.csv', index=False)
print(f'Saved {len(output)} rows to data/CDCR_heat_risk_index.csv')
print(f'Facilities: {output["cdcr_code"].nunique()}, Time periods: {output["time_period"].unique()}')

# Summary view
summary = output[output['time_period'] == 'midcentury'][
    ['cdcr_code', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']
].sort_values('risk_score', ascending=False).reset_index(drop=True)
summary.index += 1
print('\nMid-century risk ranking:')
print(summary.to_string())

## 8. PBSP ratio outlier check

In [ ]:
# PBSP has ratio_indoor_to_outdoor = 15.75 — all other facilities are < 2.0
# Check how much this outlier inflates PBSP's exposure score vs. a capped version

pbsp = df[df['cdcr_code'] == 'PBSP'].iloc[0]
print(f'PBSP ratio: {pbsp["ratio_indoor_to_outdoor"]}')
print(f'PBSP outdoor 78F days (implied): {pbsp["days_indoor_above_78f_2025"] / pbsp["ratio_indoor_to_outdoor"]:.1f}')
print(f'PBSP exposure_score (with outlier): {pbsp["exposure_score"]:.3f}')
print(f'PBSP exp_ratio (with outlier): {pbsp["exp_ratio"]:.3f}')

# What would PBSP exposure score be if ratio were capped at p95 of other facilities?
other_ratios = df.loc[df['cdcr_code'] != 'PBSP', 'ratio_indoor_to_outdoor']
p95 = other_ratios.quantile(0.95)
print(f'\n95th pctl ratio (excl. PBSP): {p95:.3f}')
print('(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)')